# TSNFA Step-by-Step: Understanding the Three Defences

This notebook walks through the Temporal Spectral Noise-Floor Adaptation (TSNFA) algorithm
step by step, visualising what happens at each stage of signal processing. You'll see:

1. How the synthetic noise model generates realistic sensor data
2. How the FFT isolates the event band (Defence 1: Band Selection)
3. How the persistence filter rejects transient spikes (Defence 2)
4. How the adaptive noise floor tracks drift (Defence 3)
5. Why time-domain methods (Zhang) fail on the same signal

**Paper:** Makovetskii & Thomsen, "Temporal Spectral Noise-Floor Adaptation for Autonomous Edge Triggering in IoT Mesh Sensor Networks," IEEE IoT Journal, 2026.

**Repository:** [github.com/Gnacode/IEEE-TSNFA](https://github.com/Gnacode/IEEE-TSNFA)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import matplotlib.gridspec as gridspec

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Simulation parameters (matching the paper)
fs = 100          # Sample rate (Hz)
L = 128           # Frame size (samples)
Tf = L / fs       # Frame duration (s) = 1.28
P0 = 1.0          # Base noise power
A_dB = 6.0        # Noise drift amplitude (dB)
T_cycle = 3600    # Noise cycle period (s) = 1 hour
SNR_dB = 18       # Event SNR (dB)
emi_amp = 0.3     # EMI amplitude as fraction of sqrt(P)
dig_amp = 2.0     # Digital burst amplitude
zeta = 6.0        # Threshold multiplier
gamma_d = 3       # Persistence filter window
gamma_a = 64      # Noise floor adaptation depth
alpha = 1 - 1/gamma_a  # EMA coefficient = 0.984

# Event band bins
bin_lo, bin_hi = 1, 6  # FFT bins covering 0.78 - 4.69 Hz

np.random.seed(42)
print(f'Frame duration: {Tf:.2f} s')
print(f'Frequency resolution: {fs/L:.3f} Hz')
print(f'Event band: bins {bin_lo}-{bin_hi} = [{bin_lo*fs/L:.2f}, {bin_hi*fs/L:.2f}] Hz')
print(f'EMA coefficient alpha = {alpha:.4f}')
print(f'SNR = {SNR_dB} dB = {10**(SNR_dB/20):.1f}x amplitude ratio')

## 1. The Noise Model

The simulation uses a multi-component noise model where the noise power $P(t)$ drifts $\pm 6$ dB over a 1-hour cycle. Let's visualise one full cycle and see how each noise component contributes.

In [ ]:
# Generate one hour of noise power variation
t_hour = np.linspace(0, 3600, 3600)
P_t = P0 * 10**((A_dB/10) * np.sin(2 * np.pi * t_hour / T_cycle))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: Power in linear scale
axes[0].plot(t_hour/60, P_t, 'b-', lw=2)
axes[0].axhline(P0, color='gray', ls='--', label=f'$P_0$ = {P0}')
axes[0].axhline(4*P0, color='red', ls=':', alpha=0.7, label=f'Peak: 4$P_0$')
axes[0].axhline(P0/4, color='blue', ls=':', alpha=0.7, label=f'Trough: $P_0$/4')
axes[0].set_xlabel('Time (minutes)')
axes[0].set_ylabel('Noise Power P(t)')
axes[0].set_title('Noise power drift over 1 hour')
axes[0].legend(fontsize=9)

# Right: Power in dB
P_dB = 10 * np.log10(P_t / P0)
axes[1].plot(t_hour/60, P_dB, 'b-', lw=2)
axes[1].axhline(0, color='gray', ls='--')
axes[1].axhline(6, color='red', ls=':', alpha=0.7, label='+6 dB')
axes[1].axhline(-6, color='blue', ls=':', alpha=0.7, label='-6 dB')
axes[1].set_xlabel('Time (minutes)')
axes[1].set_ylabel('Noise Power (dB re P₀)')
axes[1].set_title('Same drift in dB: factor of 16 from min to max')
axes[1].legend(fontsize=9)
axes[1].set_ylim(-8, 8)

plt.tight_layout()
plt.show()

print(f'P swings from {P_t.min():.3f} to {P_t.max():.3f} (ratio {P_t.max()/P_t.min():.1f}x)')

## 2. Generating One Frame of Sensor Data

Let's generate a single 128-sample frame containing all four signal components:
- Thermal noise ($w_{th}$)
- 60 Hz EMI ($w_{EMI}$)
- Digital switching burst ($w_{dig}$)
- An event signal ($s[n]$) at 2 Hz with 18 dB SNR

We'll show the frame with and without the event, so you can see why time-domain detection is hard.

In [ ]:
def generate_frame(P, include_event=False, event_freq=2.0):
    """Generate one L-sample frame with the full noise model."""
    n = np.arange(L)
    t = n / fs
    
    # Thermal noise
    w_th = np.random.randn(L) * np.sqrt(P)
    
    # 60 Hz EMI
    w_emi = emi_amp * np.sqrt(P) * np.sin(2 * np.pi * 60 * t)
    
    # Digital switching burst (intermittent)
    w_dig = np.zeros(L)
    if np.random.random() > 0.6:  # 40% chance of burst in this frame
        burst_start = np.random.randint(0, L-20)
        burst_len = np.random.randint(10, 30)
        burst_freq = np.random.uniform(800, 2000)
        w_dig[burst_start:burst_start+burst_len] = (
            dig_amp * np.sqrt(P) * np.sin(2 * np.pi * burst_freq * t[burst_start:burst_start+burst_len])
        )
    
    # Event signal (damped sinusoid at event_freq Hz)
    s = np.zeros(L)
    if include_event:
        event_amp = np.sqrt(P) * 10**(SNR_dB/20)
        decay = np.exp(-t * 0.5)  # Damping
        s = event_amp * decay * np.sin(2 * np.pi * event_freq * t)
    
    x = s + w_th + w_emi + w_dig
    return x, s, w_th, w_emi, w_dig

# Generate frames at baseline noise power
P_now = P0
np.random.seed(42)
x_noise, _, w_th, w_emi, w_dig = generate_frame(P_now, include_event=False)
np.random.seed(42)
x_event, s, _, _, _ = generate_frame(P_now, include_event=True, event_freq=2.0)

fig, axes = plt.subplots(2, 2, figsize=(14, 7))
t_frame = np.arange(L) / fs

# Individual components
axes[0,0].plot(t_frame, w_th, 'b-', alpha=0.6, lw=0.8, label='Thermal')
axes[0,0].plot(t_frame, w_emi, 'r-', alpha=0.8, lw=1.2, label='60 Hz EMI')
axes[0,0].plot(t_frame, w_dig, 'orange', alpha=0.8, lw=1, label='Digital burst')
axes[0,0].set_title('Noise components (individual)')
axes[0,0].legend(fontsize=9)
axes[0,0].set_ylabel('Amplitude')

# Event signal alone
axes[0,1].plot(t_frame, s, 'g-', lw=2, label=f'Event at 2 Hz, SNR={SNR_dB} dB')
axes[0,1].set_title('Event signal (clean)')
axes[0,1].legend(fontsize=9)

# Noise only frame
axes[1,0].plot(t_frame, x_noise, 'b-', lw=0.8)
peak_noise = np.max(np.abs(x_noise))
axes[1,0].set_title(f'Frame: noise only (peak = {peak_noise:.2f})')
axes[1,0].set_xlabel('Time (s)')
axes[1,0].set_ylabel('Amplitude')

# Noise + event frame
axes[1,1].plot(t_frame, x_event, 'b-', lw=0.8)
axes[1,1].plot(t_frame, s, 'g-', lw=1.5, alpha=0.5, label='Event (hidden inside)')
peak_event = np.max(np.abs(x_event))
axes[1,1].set_title(f'Frame: noise + event (peak = {peak_event:.2f})')
axes[1,1].set_xlabel('Time (s)')
axes[1,1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f'\nKey observation: peak amplitude is {peak_noise:.2f} (noise) vs {peak_event:.2f} (noise+event)')
print(f'A time-domain detector sees similar peaks in both cases!')

## 3. Defence 1: Spectral Band Selection (FFT)

Now let's apply the 128-point FFT and see how band selection separates the event from the noise. This is where TSNFA gains its primary advantage: the event energy is concentrated in bins 1-6 (0.78-4.69 Hz), while EMI and digital noise sit at completely different frequencies.

In [ ]:
# Compute FFT for both frames
freqs = np.fft.fftfreq(L, 1/fs)[:L//2]
spectrum_noise = np.abs(np.fft.fft(x_noise))[:L//2]
spectrum_event = np.abs(np.fft.fft(x_event))[:L//2]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, spec, title in [(axes[0], spectrum_noise, 'Noise only'),
                          (axes[1], spectrum_event, 'Noise + Event')]:
    # Full spectrum
    ax.semilogy(freqs, spec, 'b-', alpha=0.4, lw=0.8, label='Full spectrum')
    
    # Highlight event band
    band_freqs = freqs[bin_lo:bin_hi+1]
    band_mags = spec[bin_lo:bin_hi+1]
    ax.bar(band_freqs, band_mags, width=0.6, color='green', alpha=0.7, label='Event band (1-5 Hz)', zorder=5)
    
    # Mark EMI
    emi_bin = int(60 * L / fs)
    if emi_bin < len(spec):
        ax.bar(freqs[emi_bin], spec[emi_bin], width=0.6, color='red', alpha=0.7, label='60 Hz EMI', zorder=5)
    
    # TSNFA detection statistic
    X_m = np.max(band_mags)
    ax.axhline(X_m, color='green', ls='--', alpha=0.5)
    ax.annotate(f'X(m) = {X_m:.1f}', xy=(8, X_m), fontsize=10, color='green', fontweight='bold')
    
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('|X[k]| (log scale)')
    ax.set_title(title)
    ax.legend(fontsize=9, loc='upper right')
    ax.set_xlim(0, 50)
    ax.set_ylim(0.1, None)
    
    # Shade the event band region
    ax.axvspan(0.78, 4.69, alpha=0.08, color='green')

plt.tight_layout()
plt.show()

X_noise = np.max(spectrum_noise[bin_lo:bin_hi+1])
X_event = np.max(spectrum_event[bin_lo:bin_hi+1])
print(f'In-band max: {X_noise:.1f} (noise only) vs {X_event:.1f} (with event)')
print(f'Ratio: {X_event/X_noise:.1f}x — the event is clearly visible in the frequency domain!')
print(f'\nMeanwhile, a time-domain detector sees the 60 Hz EMI peak dominating everything.')

## 4. Defence 2: Persistence Filter ($\gamma_d$)

The persistence filter requires the in-band energy to stay elevated for $\gamma_d = 3$ consecutive frames. A single-frame noise spike gets averaged down to 1/3 of its peak, while a genuine multi-frame event passes through. Let's see this in action.

In [ ]:
# Simulate 30 frames: noise only, then inject an event, then noise again
n_frames = 30
event_frames = [12, 13, 14, 15]  # Event active in frames 12-15 (~5 seconds)
spike_frame = 7  # Single noise spike in frame 7

X_raw = []  # Raw max-bin values
X_bar = []  # After gamma_d mean filter
buffer = []

for f in range(n_frames):
    P_now = P0  # Constant noise for this demo
    has_event = f in event_frames
    
    # Generate frame
    x, _, _, _, _ = generate_frame(P_now, include_event=has_event, event_freq=2.5)
    
    # Inject artificial spike in frame 7
    if f == spike_frame:
        x += 8 * np.sqrt(P_now) * np.sin(2 * np.pi * 3.0 * np.arange(L)/fs) * np.exp(-np.arange(L)/fs * 5)
    
    # FFT + band selection
    spec = np.abs(np.fft.fft(x))[:L//2]
    X_m = np.max(spec[bin_lo:bin_hi+1])
    X_raw.append(X_m)
    
    # Persistence filter
    buffer.append(X_m)
    if len(buffer) > gamma_d:
        buffer.pop(0)
    X_bar.append(np.mean(buffer))

X_raw = np.array(X_raw)
X_bar = np.array(X_bar)
frames = np.arange(n_frames)

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(frames, X_raw, 'b-o', markersize=5, alpha=0.5, label='X(m) raw (before filter)', lw=1)
ax.plot(frames, X_bar, 'g-s', markersize=6, lw=2.5, label=f'X̄(m) filtered (γd={gamma_d} mean)')

# Shade event region
ax.axvspan(event_frames[0]-0.5, event_frames[-1]+0.5, alpha=0.15, color='green', label='Event active')

# Mark spike
ax.annotate('Single-frame spike\n(wind gust)', xy=(spike_frame, X_raw[spike_frame]),
            xytext=(spike_frame+2, X_raw[spike_frame]*1.1),
            arrowprops=dict(arrowstyle='->', color='red'), fontsize=10, color='red')

ax.set_xlabel('Frame number')
ax.set_ylabel('In-band magnitude')
ax.set_title('Defence 2: Persistence filter suppresses single-frame spikes, passes multi-frame events')
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f'Spike in frame {spike_frame}: raw={X_raw[spike_frame]:.1f}, filtered={X_bar[spike_frame]:.1f} (suppressed by {X_raw[spike_frame]/X_bar[spike_frame]:.1f}x)')
print(f'Event in frames {event_frames}: raw avg={X_raw[event_frames].mean():.1f}, filtered avg={X_bar[event_frames].mean():.1f} (preserved)')

## 5. Defence 3: Adaptive Noise Floor

Now let's simulate a longer run (500 frames ≈ 10 minutes) where the noise power drifts. We'll see how the gated EMA tracks the changing noise floor while keeping the threshold calibrated.

In [ ]:
# Simulate 500 frames with drifting noise
n_sim = 500
R_gate = 0.8

# Time array
t_sim = np.arange(n_sim) * Tf

# Noise power varies
P_sim = P0 * 10**((A_dB/10) * np.sin(2 * np.pi * t_sim / T_cycle))

# Inject events at specific frames
event_at = [100, 101, 102, 103,    # Event 1
            250, 251, 252,          # Event 2 (during high noise)
            400, 401, 402, 403]     # Event 3 (during low noise)

# Arrays to track
X_raw_sim = np.zeros(n_sim)
X_bar_sim = np.zeros(n_sim)
N_hat = np.zeros(n_sim)
Theta = np.zeros(n_sim)
R_sim = np.zeros(n_sim)
triggers = np.zeros(n_sim, dtype=bool)

# Initialise
N_hat[0] = np.sqrt(P0) * np.sqrt(L)  # Initial noise floor estimate
filt_buf = []

for f in range(n_sim):
    has_event = f in event_at
    x, _, _, _, _ = generate_frame(P_sim[f], include_event=has_event, event_freq=2.5)
    
    # Stage 1: FFT + band selection
    spec = np.abs(np.fft.fft(x))[:L//2]
    X_m = np.max(spec[bin_lo:bin_hi+1])
    X_raw_sim[f] = X_m
    
    # Stage 2: Persistence filter
    filt_buf.append(X_m)
    if len(filt_buf) > gamma_d:
        filt_buf.pop(0)
    X_bar_sim[f] = np.mean(filt_buf)
    
    # Stage 3: Threshold
    N_prev = N_hat[f-1] if f > 0 else N_hat[0]
    Theta[f] = zeta * N_prev
    
    # Stage 4: Trigger
    R_sim[f] = X_bar_sim[f] / Theta[f] if Theta[f] > 0 else 0
    triggers[f] = R_sim[f] > 1.0
    
    # Stage 5: Noise floor update (gated)
    if R_sim[f] < R_gate:
        N_hat[f] = alpha * N_prev + (1-alpha) * X_bar_sim[f]
    else:
        N_hat[f] = N_prev

# Plot results
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Top: Signal and threshold
ax = axes[0]
ax.plot(t_sim/60, X_bar_sim, 'b-', lw=0.8, alpha=0.6, label='X̄(m) filtered')
ax.plot(t_sim/60, Theta, 'r-', lw=2, label='Threshold Θ = ζ × N̂')
ax.plot(t_sim/60, N_hat, 'orange', lw=2, ls='--', label='Noise floor N̂')
trig_idx = np.where(triggers)[0]
ax.scatter(t_sim[trig_idx]/60, X_bar_sim[trig_idx], color='green', s=80, zorder=5, marker='^', label='Triggers')
ax.set_ylabel('Magnitude')
ax.set_title('TSNFA: Adaptive threshold tracks noise drift, triggers only on real events')
ax.legend(fontsize=9, loc='upper right')

# Middle: Noise power
ax = axes[1]
ax.plot(t_sim/60, 10*np.log10(P_sim/P0), 'purple', lw=2)
ax.set_ylabel('Noise power (dB re P₀)')
ax.set_title('Noise power drift')
ax.set_ylim(-8, 8)
for ef in event_at:
    if ef == event_at[0] or (ef > 0 and ef-1 not in event_at):
        ax.axvline(t_sim[ef]/60, color='green', alpha=0.3, lw=3)

# Bottom: Detection ratio R
ax = axes[2]
ax.plot(t_sim/60, R_sim, 'b-', lw=0.8)
ax.axhline(1.0, color='red', ls='-', lw=2, alpha=0.7, label='Trigger threshold (R=1.0)')
ax.axhline(0.8, color='orange', ls='--', lw=1.5, alpha=0.7, label='Gate threshold (R=0.8)')
ax.fill_between(t_sim/60, 0, 0.8, alpha=0.05, color='green', label='NF update zone')
ax.set_ylabel('R(m)')
ax.set_xlabel('Time (minutes)')
ax.set_title('Detection ratio R: below 0.8 = update noise floor, above 1.0 = trigger')
ax.legend(fontsize=9)
ax.set_ylim(0, max(2, R_sim.max()*1.1))

plt.tight_layout()
plt.show()

n_true_events = len(set(event_at))
n_triggers = triggers.sum()
n_true_triggers = sum(1 for f in range(n_sim) if triggers[f] and f in event_at)
print(f'Events injected in {n_true_events} frames, triggers fired: {n_triggers}')
print(f'True positives: {n_true_triggers}, False positives: {n_triggers - n_true_triggers}')

## 6. Why Zhang's Time-Domain Method Fails

Let's run the same signal through Zhang's time-domain detector (Algorithm 3) and see why it produces both false positives and false negatives.

In [ ]:
# Re-run with Zhang's method on the SAME frames
beta_zhang = 0.95
np.random.seed(42)  # Same random seed = same noise realisation

NZ = np.zeros(n_sim)
XZ = np.zeros(n_sim)
RZ = np.zeros(n_sim)
ThetaZ = np.zeros(n_sim)
triggers_zhang = np.zeros(n_sim, dtype=bool)

NZ[0] = np.sqrt(P0) * 3  # Initial estimate

for f in range(n_sim):
    has_event = f in event_at
    x, _, _, _, _ = generate_frame(P_sim[f], include_event=has_event, event_freq=2.5)
    
    # Zhang: time-domain peak (NO FFT)
    XZ[f] = np.max(np.abs(x))
    
    # Threshold
    NZ_prev = NZ[f-1] if f > 0 else NZ[0]
    ThetaZ[f] = zeta * NZ_prev
    
    # Trigger
    RZ[f] = XZ[f] / ThetaZ[f] if ThetaZ[f] > 0 else 0
    triggers_zhang[f] = RZ[f] > 1.0
    
    # Noise floor update
    if RZ[f] < 0.8:
        NZ[f] = beta_zhang * NZ_prev + (1-beta_zhang) * XZ[f]
    else:
        NZ[f] = NZ_prev

# Compare
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# TSNFA
ax = axes[0]
ax.plot(t_sim/60, X_bar_sim, 'b-', lw=0.8, alpha=0.5)
ax.plot(t_sim/60, Theta, 'r-', lw=2, alpha=0.7)
trig_idx = np.where(triggers)[0]
fp_tsnfa = [f for f in trig_idx if f not in event_at]
tp_tsnfa = [f for f in trig_idx if f in event_at]
ax.scatter(t_sim[tp_tsnfa]/60, X_bar_sim[tp_tsnfa], color='green', s=100, zorder=5, marker='^', label=f'TP: {len(tp_tsnfa)}')
ax.scatter(t_sim[fp_tsnfa]/60, X_bar_sim[fp_tsnfa], color='red', s=100, zorder=5, marker='x', label=f'FP: {len(fp_tsnfa)}')
ax.set_title(f'TSNFA (Algorithm 1): {len(tp_tsnfa)} TP, {len(fp_tsnfa)} FP — spectral filtering isolates events', fontsize=12)
ax.set_ylabel('In-band magnitude')
ax.legend(fontsize=10)

# Zhang
ax = axes[1]
ax.plot(t_sim/60, XZ, 'b-', lw=0.8, alpha=0.5)
ax.plot(t_sim/60, ThetaZ, 'r-', lw=2, alpha=0.7)
trig_z_idx = np.where(triggers_zhang)[0]
fp_zhang = [f for f in trig_z_idx if f not in event_at]
tp_zhang = [f for f in trig_z_idx if f in event_at]
ax.scatter(t_sim[tp_zhang]/60, XZ[tp_zhang], color='green', s=100, zorder=5, marker='^', label=f'TP: {len(tp_zhang)}')
ax.scatter(t_sim[fp_zhang]/60, XZ[fp_zhang], color='red', s=100, zorder=5, marker='x', label=f'FP: {len(fp_zhang)}')
ax.set_title(f'Zhang (Algorithm 3): {len(tp_zhang)} TP, {len(fp_zhang)} FP — composite noise inflates threshold', fontsize=12)
ax.set_ylabel('Time-domain peak')
ax.set_xlabel('Time (minutes)')
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

## 7. What Each Method "Sees"

Finally, let's take a single event frame and show what detection statistic each algorithm computes from the same 128 samples.

In [ ]:
# Generate one event frame
np.random.seed(99)
x_demo, s_demo, w_th_demo, w_emi_demo, w_dig_demo = generate_frame(P0, include_event=True, event_freq=2.5)
spec_demo = np.abs(np.fft.fft(x_demo))[:L//2]

fig = plt.figure(figsize=(14, 10))
gs = gridspec.GridSpec(3, 2, hspace=0.4, wspace=0.3)

# Time domain
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(np.arange(L)/fs, x_demo, 'b-', lw=0.8, label='Composite signal')
ax1.plot(np.arange(L)/fs, s_demo, 'g-', lw=2, alpha=0.6, label='Event (hidden)')
ax1.axhline(np.max(np.abs(x_demo)), color='red', ls=':', alpha=0.5, label=f'Peak = {np.max(np.abs(x_demo)):.1f}')
ax1.set_title('Same 128 samples fed to all algorithms', fontsize=13)
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Amplitude')
ax1.legend(fontsize=9)

# TSNFA view
ax2 = fig.add_subplot(gs[1, 0])
ax2.bar(freqs[bin_lo:bin_hi+1], spec_demo[bin_lo:bin_hi+1], width=0.5, color='green', alpha=0.8)
ax2.set_title(f'TSNFA sees: max band = {np.max(spec_demo[bin_lo:bin_hi+1]):.1f}', fontsize=12, color='green')
ax2.set_xlabel('Frequency (Hz)')
ax2.set_ylabel('|X[k]|')
ax2.set_xlim(0, 6)

# Zhang view
ax3 = fig.add_subplot(gs[1, 1])
ax3.plot(np.arange(L)/fs, np.abs(x_demo), 'r-', lw=0.5)
ax3.axhline(np.max(np.abs(x_demo)), color='red', lw=2)
ax3.set_title(f'Zhang sees: peak = {np.max(np.abs(x_demo)):.1f} (includes EMI)', fontsize=12, color='red')
ax3.set_xlabel('Time (s)')
ax3.set_ylabel('|x[n]|')

# DEDaR view
ax4 = fig.add_subplot(gs[2, 0])
energy = np.sum(x_demo**2)
ax4.bar(['Frame energy'], [energy], color='#e65100', alpha=0.8)
ax4.set_title(f'DEDaR sees: total energy = {energy:.0f} (all frequencies)', fontsize=12, color='#e65100')
ax4.set_ylabel('Σ|x[n]|²')

# Summary comparison
ax5 = fig.add_subplot(gs[2, 1])
methods = ['TSNFA', 'Zhang', 'STFT', 'DEDaR']
what_they_see = [
    np.max(spec_demo[bin_lo:bin_hi+1]),
    np.max(np.abs(x_demo)),
    np.max(spec_demo[bin_lo:bin_hi+1]),  # Same as TSNFA
    np.sum(x_demo**2) / 100,  # Scaled for display
]
colors = ['#2e7d32', '#d32f2f', '#1565c0', '#e65100']
bars = ax5.barh(methods, what_they_see, color=colors, alpha=0.8)
ax5.set_title('Detection statistic (higher = more sensitive)', fontsize=12)
ax5.set_xlabel('Value (arbitrary scale)')

plt.tight_layout()
plt.show()

print('Key insight: TSNFA and STFT see the event clearly in the frequency domain.')
print('Zhang sees the composite peak dominated by EMI.')
print('DEDaR sees total broadband energy, drowning the event in noise from all frequencies.')

## Summary

This notebook demonstrated the three defences that make TSNFA unique:

| Defence | What it does | What it rejects |
|---------|-------------|----------------|
| **Band selection** (FFT) | Isolates 1-5 Hz event band | 60 Hz EMI, digital switching, out-of-band thermal |
| **Persistence** (γd mean) | Requires multi-frame elevation | Single-frame wind gusts, ADC glitches |
| **Adaptive NF** (gated EMA) | Tracks ±6 dB noise drift | Stale thresholds that cause FP during high-noise phases |

No other method in the comparison implements all three. The full 200-node, 24-hour Monte Carlo
simulation confirms: **100% detection rate, 0 false positives**.

Run the full simulation with:
```bash
cd simulation
python IOTfulltest4-withNoise4.py --nodes 200 --preset ACCURATE
```

Interactive dashboard: [gnacode.github.io/IEEE-TSNFA](https://gnacode.github.io/IEEE-TSNFA/)